Swarm implements a team in which agents can hand off task to other agents based on their capabilities. It is a multi-agent design pattern first introduced by OpenAI in Swarm. The key idea is to let agent delegate tasks to other agents using a special tool call, while all agents share the same message context. This enables agents to make local decisions about task planning, rather than relying on a central orchestrator such as in SelectorGroupChat.

At its core, the Swarm team is a group chat where agents take turn to generate a response. Similar to SelectorGroupChat and RoundRobinGroupChat, participant agents broadcast their responses so all agents share the same message context.

Different from the other two group chat teams, at each turn, the speaker agent is selected based on the most recent HandoffMessage message in the context. This naturally requires each agent in the team to be able to generate HandoffMessage to signal which other agents that it hands off to.

##Stock Research Example

This system is designed to perform stock research tasks by leveraging four agents:

Planner: The central coordinator that delegates specific tasks to specialized agents based on their expertise. The planner ensures that each agent is utilized efficiently and oversees the overall workflow.

Financial Analyst: A specialized agent responsible for analyzing financial metrics and stock data using tools such as get_stock_data.

News Analyst: An agent focused on gathering and summarizing recent news articles relevant to the stock, using tools such as get_news.

Writer: An agent tasked with compiling the findings from the stock and news analysis into a cohesive final report.

##Workflow ->
The Planner initiates the research process by delegating tasks to the appropriate agents in a step-by-step manner.

Each agent performs its task independently and appends their work to the shared message thread/history. Rather than directly returning results to the planner, all agents contribute to and read from this shared message history. When agents generate their work using the LLM, they have access to this shared message history, which provides context and helps track the overall progress of the task.

Once an agent completes its task, it hands off control back to the planner.

The process continues until the planner determines that all necessary tasks have been completed and decides to terminate the workflow.

In [1]:
from typing import List, Any, Dict
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_agentchat.messages import HandoffMessage
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination, MaxMessageTermination
from autogen_agentchat.teams import Swarm
import os
import sys
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
load_dotenv()

True

#Tools

In [2]:
async def get_stock_data(symbol: str) -> Dict[str, Any]:
    """Get stock market data for a given symbol"""
    return {"price": 180.25, "volume": 1000000, "pe_ratio": 65.4, "market_cap": "700B"}


async def get_news(query: str) -> List[Dict[str, str]]:
    """Get recent news articles about a company"""
    return [
        {
            "title": "Tesla Expands Cybertruck Production",
            "date": "2024-03-20",
            "summary": "Tesla ramps up Cybertruck manufacturing capacity at Gigafactory Texas, aiming to meet strong demand.",
        },
        {
            "title": "Tesla FSD Beta Shows Promise",
            "date": "2024-03-19",
            "summary": "Latest Full Self-Driving beta demonstrates significant improvements in urban navigation and safety features.",
        },
        {
            "title": "Model Y Dominates Global EV Sales",
            "date": "2024-03-18",
            "summary": "Tesla's Model Y becomes best-selling electric vehicle worldwide, capturing significant market share.",
        },
    ]


In [3]:
model_client = OpenAIChatCompletionClient(
    model = 'gpt-3.5-turbo'
)

In [4]:
planner = AssistantAgent(
    "planner",
    model_client=model_client,
    handoffs=["financial_analyst", "news_analyst", "writer"],
    system_message="""You are a research planning coordinator.
    Coordinate market research by delegating to specialized agents:
    - Financial Analyst: For stock data analysis
    - News Analyst: For news gathering and analysis
    - Writer: For compiling final report
    Always send your plan first, then handoff to appropriate agent.
    Always handoff to a single agent at a time.
    Use TERMINATE after you receive output from the writer agent""",
)

In [5]:
financial_analyst = AssistantAgent(
    "financial_analyst",
    model_client=model_client,
    handoffs=["planner"],
    tools=[get_stock_data],
    system_message="""You are a financial analyst.
    Analyze stock market data using the get_stock_data tool.
    Provide insights on financial metrics.
    Always handoff back to planner when analysis is complete.""",
)

In [6]:
news_analyst = AssistantAgent(
    "news_analyst",
    model_client=model_client,
    handoffs=["planner"],
    tools=[get_news],
    system_message="""You are a news analyst.
    Gather and analyze relevant news using the get_news tool.
    Summarize key market insights from news.
    Always handoff back to planner when analysis is complete.""",
)

In [7]:
writer = AssistantAgent(
    "writer",
    model_client=model_client,
    handoffs=["planner"],
    system_message="""You are a financial report writer.
    Compile research findings into clear, concise reports.
    Always handoff back to planner when writing is complete.""",
)

In [8]:
text_termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(15)

In [9]:
research_team = Swarm(
    participants=[planner, financial_analyst, news_analyst, writer], termination_condition=text_termination
)

In [10]:
task = "Conduct market research for TSLA stock"
await Console(research_team.run_stream(task=task))
await model_client.close()

---------- TextMessage (user) ----------
Conduct market research for TSLA stock
---------- ToolCallRequestEvent (planner) ----------
[FunctionCall(id='call_QQ1OiCDuHVjNvA0iqxuGruWx', arguments='{}', name='transfer_to_financial_analyst')]
---------- ToolCallExecutionEvent (planner) ----------
[FunctionExecutionResult(content='Transferred to financial_analyst, adopting the role of financial_analyst immediately.', name='transfer_to_financial_analyst', call_id='call_QQ1OiCDuHVjNvA0iqxuGruWx', is_error=False)]
---------- HandoffMessage (planner) ----------
Transferred to financial_analyst, adopting the role of financial_analyst immediately.
---------- ToolCallRequestEvent (financial_analyst) ----------
[FunctionCall(id='call_Lrybi26pw82eFXTardy6RFb4', arguments='{"symbol":"TSLA"}', name='get_stock_data')]
---------- ToolCallExecutionEvent (financial_analyst) ----------
[FunctionExecutionResult(content="{'price': 180.25, 'volume': 1000000, 'pe_ratio': 65.4, 'market_cap': '700B'}", name='get_